<a href="https://colab.research.google.com/github/muhammad-bin-nasir/Sentiment-Analysis/blob/main/SentimentAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define your project directory paths
# IMPORTANT: Change 'Your_Project_Folder' to your actual folder name
base_path = '/content/drive/My Drive/Sentiment analysis'
audio_folder = os.path.join(base_path, 'Audio')
labels_csv = os.path.join(base_path, 'labels_fixed.csv')

# Create Output/MFCC directory if it doesn't exist
output_dir = os.path.join(base_path, 'Output', 'MFCC')
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Output will be saved to: {output_dir}")

Mounted at /content/drive
✅ Output will be saved to: /content/drive/My Drive/Sentiment analysis/Output/MFCC


<h1>Wav2Vec2

Model

In [4]:
!pip install evaluate joblib

import os
import pandas as pd
import numpy as np
import torch
import librosa
import soundfile as sf
import joblib
from datasets import Dataset
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.preprocessing import LabelEncoder
from evaluate import load as load_metric

# ============================================================
# Step 1: Cloud Configurations
# ============================================================
# Ensure these paths match your Google Drive setup
base_path = '/content/drive/My Drive/Sentiment analysis'
audio_folder = os.path.join(base_path, 'Audio')
labels_csv = os.path.join(base_path, 'labels_fixed.csv')

output_drive_dir = os.path.join(base_path, "Output", "Wav2Vec2_Finetuned")
os.makedirs(output_drive_dir, exist_ok=True)

MODEL_NAME = "facebook/wav2vec2-base"

# ============================================================
# Step 2: Load, Encode, and Save Labels
# ============================================================
df = pd.read_csv(labels_csv)
label_encoder = LabelEncoder()
df["label_id"] = label_encoder.fit_transform(df["label"])

id2label = {i: l for i, l in enumerate(label_encoder.classes_)}
label2id = {l: i for i, l in enumerate(label_encoder.classes_)}

# Save the encoder for future inference right away
joblib.dump(label_encoder, os.path.join(output_drive_dir, "label_encoder.pkl"))
print(f"✅ Label encoder saved securely to {output_drive_dir}")

# ============================================================
# Step 3: Load Processor + Model
# ============================================================
processor = Wav2Vec2Processor.from_pretrained(MODEL_NAME)
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_encoder.classes_),
    label2id=label2id,
    id2label=id2label
)

# ============================================================
# Step 4: Dataset Processing & 70-15-15 Split
# ============================================================
def load_audio(batch):
    file_path = os.path.join(audio_folder, batch["filename"])
    try:
        speech, sr = sf.read(file_path)
        if len(speech.shape) > 1:
            speech = np.mean(speech, axis=1)
        if sr != 16000:
            speech = librosa.resample(speech, orig_sr=sr, target_sr=16000)

        batch["input_values"] = processor(
            speech,
            sampling_rate=16000,
            max_length=80000,
            truncation=True
        ).input_values[0]
        batch["labels"] = batch["label_id"]
    except Exception as e:
        batch["input_values"] = [0.0] * 1000
        batch["labels"] = -1
    return batch

dataset = Dataset.from_pandas(df)
dataset = dataset.map(load_audio, remove_columns=dataset.column_names, desc="Processing Audio")
dataset = dataset.filter(lambda x: x["labels"] != -1)

# Perform 70/15/15 Split
train_testvalid = dataset.train_test_split(test_size=0.3)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5)

train_ds = train_testvalid['train'] # 70%
val_ds = test_valid['train']        # 15%
test_ds = test_valid['test']        # 15%

print(f"📊 Dataset Split: Train({len(train_ds)}) | Val({len(val_ds)}) | Test({len(test_ds)})")

# ============================================================
# Step 5: Collator & Metrics
# ============================================================
def collate_fn(batch):
    input_values = [x["input_values"] for x in batch]
    labels = [x["labels"] for x in batch]
    inputs = processor.pad({"input_values": input_values}, return_tensors="pt", padding=True)
    inputs["labels"] = torch.tensor(labels)
    return inputs

accuracy_metric = load_metric("accuracy")

def compute_metrics(pred):
    preds = np.argmax(pred.predictions, axis=1)
    return accuracy_metric.compute(predictions=preds, references=pred.label_ids)

# ============================================================
# Step 6: Training Setup (2x Epochs & Checkpoint Management)
# ============================================================
epochs = 20
total_steps = (len(train_ds) // 16) * epochs
warmup_steps = int(total_steps * 0.1)

training_args = TrainingArguments(
    output_dir=output_drive_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,                  # Automatically deletes older checkpoints
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=epochs,             # Increased to 20
    warmup_steps=warmup_steps,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    disable_tqdm=False                   # Ensures live progress bars are visible
)

# ============================================================
# Step 7: Trainer
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,                # Validating on the 15% validation set
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]
)

# Train the model
print("🚀 Starting Wav2Vec2 Fine-tuning...")
trainer.train()

# ============================================================
# Step 8: Final Save
# ============================================================
trainer.save_model(output_drive_dir)
processor.save_pretrained(output_drive_dir)
print(f"🎉 Complete! Final model and encoder saved to {output_drive_dir}")

✅ Label encoder saved securely to /content/drive/My Drive/Sentiment analysis/Output/Wav2Vec2_Finetuned


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.bias              | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Processing Audio:   0%|          | 0/12438 [00:00<?, ? examples/s]

Filter:   0%|          | 0/12438 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

🎉 Complete! Final model and encoder saved to /content/drive/My Drive/Sentiment analysis/Output/Wav2Vec2_Finetuned


<h1>Git Upload

In [2]:
import os

# --- 1. Configuration ---
GITHUB_TOKEN = "###"
USERNAME = "muhammad-bin-nasir"
REPO_NAME = "Sentiment-Analysis"
EMAIL = "notsogoofybro@gmail.com"

# --- 2. Setup Git & LFS ---
!git config --global user.name "{USERNAME}"
!git config --global user.email "{EMAIL}"
!apt-get install git-lfs
!git lfs install

# --- 3. Clone and Prepare ---
repo_url = f"https://{GITHUB_TOKEN}@github.com/{USERNAME}/{REPO_NAME}.git"
!git clone {repo_url}
%cd {REPO_NAME}

# Track large files so GitHub doesn't reject them
!git lfs track "*.bin"
!git lfs track "*.safetensors"
!git add .gitattributes

# --- 4. Copy Files from Drive ---
# Replace this with your actual Wav2Vec2 output folder on Drive
drive_output_path = "/content/drive/My Drive/Sentiment analysis/Output/Wav2Vec2_Finetuned/"

# Copy everything (Model, Processor, and Label Encoder)
!cp -r "{drive_output_path}"* .

# --- 5. Commit and Push ---
!git add .
!git commit -m "Direct upload of Wav2Vec2 model from Google Drive"
!git push origin main

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
Git LFS initialized.
Cloning into 'Sentiment-Analysis'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 18 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 87.54 KiB | 2.30 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/Sentiment-Analysis
Tracking "*.bin"
Tracking "*.safetensors"
[main 4b231f6] Direct upload of Wav2Vec2 model from Google Drive
 19 files changed, 7957 insertions(+)
 create mode 100644 .gitattributes
 create mode 100644 checkpoint-10355/config.json
 create mode 100644 checkpoint-10355/model.safetensors
 create mode 100644 checkpoint-10355/optimizer.pt
 create mode 100644 checkpoint-10355/processor_config.json

<h1>HuggingFace

In [6]:
from huggingface_hub import login, HfApi, create_repo

# 1. Configuration
# Paste your token here (Ensure it has 'WRITE' access)
HF_TOKEN = "###"
repo_id = "muhammadbn727/wav2vec2-auditory-sentiment-analysis"
# Path to your final model folder on Drive
drive_output_path = "/content/drive/My Drive/Sentiment analysis/Output/Wav2Vec2_Finetuned/"

# 2. Login
login(token=HF_TOKEN)

# 3. Initialize API
api = HfApi()

# 4. Create the Repository (Handles the 404 error)
print(f"🛠️ Ensuring repository {repo_id} exists...")
try:
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print("✅ Repository is ready.")
except Exception as e:
    print(f"ℹ️ Note: {e}")

# 5. Push the entire folder
print(f"🚀 Uploading folder from Drive to Hugging Face...")
api.upload_folder(
    folder_path=drive_output_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Initial upload of fine-tuned Wav2Vec2 model"
)

print(f"\n🎉 Success! View your model here: https://huggingface.co/{repo_id}")

🛠️ Ensuring repository muhammadbn727/wav2vec2-auditory-sentiment-analysis exists...
✅ Repository is ready.
🚀 Uploading folder from Drive to Hugging Face...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...t-10355/model.safetensors:   0%|          |  551kB /  378MB            

  ...netuned/model.safetensors:   0%|          |  157kB /  378MB            

...

  ...kpoint-10355/scheduler.pt:   7%|7         |   109B / 1.47kB            

  ...t-10355/training_args.bin:   7%|7         |   388B / 5.20kB            

  ...netuned/label_encoder.pkl:   7%|7         |  40.0B /   538B            

  ...netuned/training_args.bin:   7%|7         |   388B / 5.20kB            


🎉 Success! View your model here: https://huggingface.co/muhammadbn727/wav2vec2-auditory-sentiment-analysis


<h1>Local use

Script to download the model

In [ ]:
# First, they need to install this: pip install huggingface_hub
from huggingface_hub import snapshot_download

# Your public repository
repo_id = "muhammadbn727/wav2vec2-auditory-sentiment-analysis"

# The local folder where the model will be saved
local_dir = "./sentiment_model_offline"

print(f"Downloading model from {repo_id}...")

# This downloads everything needed to run the model locally
snapshot_download(repo_id=repo_id, local_dir=local_dir)

print(f"✅ Download complete! Model saved to {local_dir}")

Use it for sentimental analysis

In [ ]:
# They will need these installed: pip install torch transformers librosa
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

# 1. Point to the local folder where the model was downloaded
model_path = "./sentiment_model_offline"

print("Loading local model and processor...")
processor = Wav2Vec2Processor.from_pretrained(model_path)
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_path)

def analyze_audio_emotion(audio_file_path):
    """
    Loads an audio file, processes it, and returns the predicted emotion.
    """
    # 2. Load and resample audio to 16,000 Hz
    # librosa.load automatically converts stereo to mono and resamples
    speech, sr = librosa.load(audio_file_path, sr=16000)

    # 3. Prepare the audio for the model
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)

    # 4. Pass the audio through the neural network
    with torch.no_grad():
        logits = model(**inputs).logits

    # 5. Calculate probabilities and find the highest score
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    predicted_id = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_id].item()

    # 6. Translate the ID back to the text label (e.g., "Happy", "Angry")
    # This works because we saved id2label during your training phase
    emotion = model.config.id2label[predicted_id]

    return emotion, confidence

# --- How to use it ---
# The user just puts the path to their audio file here
test_file = "user_audio_recording.wav"

try:
    print(f"Analyzing {test_file}...")
    emotion, score = analyze_audio_emotion(test_file)
    print(f"🎭 Predicted Emotion: {emotion} ({score:.2%} confidence)")
except FileNotFoundError:
    print(f"❌ Error: Could not find the file '{test_file}'")
except Exception as e:
    print(f"❌ An error occurred: {e}")

<h1>For remote/direct HF work

In [ ]:
# Prerequisites: pip install torch transformers librosa
import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2ForSequenceClassification

# 1. Define your public Hugging Face repository
REPO_ID = "muhammadbn727/wav2vec2-auditory-sentiment-analysis"

print("Loading model directly from Hugging Face (this downloads ~380MB on the first run)...")

# 2. Load the processor and model directly from the Hub
processor = Wav2Vec2Processor.from_pretrained(REPO_ID)
model = Wav2Vec2ForSequenceClassification.from_pretrained(REPO_ID)

def analyze_audio_emotion(audio_file_path):
    """
    Loads an audio file, resamples it to 16kHz, and returns the predicted emotion.
    """
    try:
        # librosa automatically handles stereo to mono and resampling
        speech, sr = librosa.load(audio_file_path, sr=16000)
    except Exception as e:
        return f"Error loading audio: {e}", 0.0

    # Prepare the audio tensor for the model
    inputs = processor(speech, sampling_rate=16000, return_tensors="pt", padding=True)

    # Pass the audio through the network
    with torch.no_grad():
        logits = model(**inputs).logits

    # Calculate probabilities
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    predicted_id = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][predicted_id].item()

    # Translate the ID back to the text label
    emotion = model.config.id2label[predicted_id]

    return emotion, confidence

# --- Example Usage ---
# The user simply provides a path to their own audio file
test_file = "sample_recording.wav" # Replace with a real audio file path

print(f"Analyzing '{test_file}'...")
emotion, score = analyze_audio_emotion(test_file)

if score > 0:
    print(f"🎭 Predicted Emotion: {emotion} ({score:.2%} confidence)")
else:
    print(emotion) # Prints the error message if file loading failed